# Foosball SAC Agent — Kaggle Training
Train a Soft Actor-Critic (SAC) agent to play foosball using MuJoCo physics simulation and Stable-Baselines3.
Run all cells top-to-bottom. Trained models are saved to `/kaggle/working/models/` and can be downloaded as a zip archive at the end.


## 1. Install ALL Dependencies

In [ ]:
import subprocess, sys, os

def run(cmd): subprocess.check_call(cmd)
def pip(*packages): subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])

# System packages for headless rendering
run(["apt-get", "install", "-y", "-q", "xvfb", "libgl1-mesa-glx", "libosmesa6", "libglfw3"])

# Python packages — mujoco MUST be installed before any import
pip("mujoco", "stable-baselines3[extra]", "gymnasium", "shimmy>=0.2.0", "pyvirtualdisplay", "glfw")

# Clone repo
REPO_URL = "https://github.com/carlkaziboni/Foosball_CU.git"
REPO_DIR = "/kaggle/working/Foosball_CU"
if not os.path.exists(REPO_DIR):
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
else:
    run(["git", "-C", REPO_DIR, "pull"])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Cell 1 done ✓")


## 2. Headless Setup + GLFW Patch
Sets headless rendering env vars, starts a virtual display, and patches the GLFW mixin **before** `FoosballEnv` is ever imported.

In [ ]:
import os, sys, warnings

# Set headless env vars BEFORE any mujoco/glfw import
os.environ["MUJOCO_GL"]         = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"

from pyvirtualdisplay import Display
_display = Display(visible=False, size=(1024, 768))
_display.start()
os.environ["DISPLAY"] = ":0"
print("Virtual display started ✓")

warnings.filterwarnings("ignore", message=".*upgrade to Gymnasium.*")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="gym")

REPO_DIR  = "/kaggle/working/Foosball_CU"
MODEL_DIR = "/kaggle/working/models"
LOG_DIR   = "/kaggle/working/logs"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR,   exist_ok=True)

# Patch GLFW mixin to no-op BEFORE FoosballEnv is imported
import ai_agents.v2.gym.mujoco_table_render_mixin as _mixin_module

class _HeadlessMixin:
    def __init__(self):
        self.viewer = None
        self.window = None
    def _initialize_glfw(self): pass
    def render(self, mode="human"): pass
    def close(self): pass

_mixin_module.MujocoTableRenderMixin = _HeadlessMixin

print(f"GLFW mixin patched ✓")
print(f"MUJOCO_GL  = {os.environ['MUJOCO_GL']}")
print(f"Model dir  : {MODEL_DIR}")
print(f"Log dir    : {LOG_DIR}")
print("Cell 2 done ✓")


## 3. Environment Factory

In [ ]:
# Mixin is already patched — safe to import FoosballEnv now
from stable_baselines3.common.monitor import Monitor
from ai_agents.v2.gym.full_information_protagonist_antagonist_gym import FoosballEnv

def sac_foosball_env_factory(x=None):
    env = FoosballEnv(antagonist_model=None)
    env = Monitor(env, filename=os.path.join(LOG_DIR, "monitor"))
    return env

# Sanity check
_test_env = sac_foosball_env_factory()
print("Observation space:", _test_env.observation_space)
print("Action space     :", _test_env.action_space)
_test_env.close()
del _test_env
print("Cell 3 done ✓")


## 4. Define StableSACAgent
Uses a subclass (not monkey-patching) to avoid recursion errors on re-run, uses a stable `[512, 512, 256]` network, and disables gSDE to prevent NaN explosion.

In [ ]:
import torch
from stable_baselines3 import SAC
from ai_agents.common.train.impl.sac_agent import SACFoosballAgent

_true_init = SACFoosballAgent.__dict__["__init__"]

class StableSACAgent(SACFoosballAgent):
    """SAC agent with stable net_arch and use_sde=False baked in at model creation."""

    def __init__(self, id, env=None,
                 log_dir=LOG_DIR,
                 model_dir=MODEL_DIR,
                 policy_kwargs=None):
        if policy_kwargs is None:
            policy_kwargs = dict(net_arch=[512, 512, 256])
        _true_init(self, id=id, env=env,
                   log_dir=log_dir, model_dir=model_dir,
                   policy_kwargs=policy_kwargs)

    def _make_sac(self, tensorboard_log=None):
        """Create a SAC model with use_sde=False — prevents CUDA/CPU device mismatch."""
        kwargs = dict(
            policy="MlpPolicy",
            env=self.env,
            policy_kwargs=self.policy_kwargs,
            device=self.device,
            learning_rate=0.001,
            buffer_size=50000,
            learning_starts=500,
            batch_size=128,
            tau=0.01,
            gamma=0.95,
            train_freq=1,
            gradient_steps=2,
            ent_coef="auto",
            target_entropy="auto",
            use_sde=False,          # ← must be False; True causes CUDA/CPU mismatch
            verbose=1,
        )
        if tensorboard_log:
            kwargs["tensorboard_log"] = tensorboard_log
        return SAC(**kwargs)

    def initialize_agent(self):
        try:
            self.load()
            print(f"Agent {self.id} initialized from saved model.")
        except Exception:
            print(f"Agent {self.id} could not load model. Initializing new model.")
            self.model = self._make_sac()
            print(f"Agent {self.id} initialized (use_sde=False).")

    def learn(self, total_timesteps):
        if self.model is None:
            tb_log = f"{LOG_DIR}/sac_agent_{self.id}"
            self.model = self._make_sac(tensorboard_log=tb_log)

        callback = self.create_callback(self.env)
        tb_log_name = f"run_{self.id}"
        print(f"Agent {self.id} starting learning for {total_timesteps} timesteps...")
        self.model.learn(
            total_timesteps=total_timesteps,
            callback=callback,
            tb_log_name=tb_log_name,
            progress_bar=True,
            log_interval=10,
        )

# GPU info
if torch.cuda.is_available():
    print(f"Training device: CUDA — {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    print("Training device: Apple MPS")
else:
    print("Training device: CPU")

print("Cell 4 done ✓")


## 5. Agent Manager + Engine

In [ ]:
from ai_agents.common.train.impl.generic_agent_manager import GenericAgentManager
from ai_agents.common.train.impl.single_player_training_engine import SinglePlayerTrainingEngine

agent_manager = GenericAgentManager(
    num_agents=1,
    environment_generator=sac_foosball_env_factory,
    agent_class=StableSACAgent,   # subclass — no recursion risk
)
agent_manager.initialize_training_agents()
agent_manager.initialize_frozen_best_models()

engine = SinglePlayerTrainingEngine(
    agent_manager=agent_manager,
    environment_generator=sac_foosball_env_factory,
)

print("Agent manager and training engine ready ✓")
print("Cell 5 done ✓")


## 6. Training Configuration
Adjust these values before running. Kaggle P100 has a **≈30 hr/week quota** — defaults stay well within that.

In [ ]:
total_epochs    = 10      # number of training epochs
epoch_timesteps = 2_000   # timesteps per epoch  →  10 × 2 000 = 20 000 total
cycle_timesteps = 500     # inner cycle length

total_timesteps = total_epochs * epoch_timesteps

print("=" * 50)
print(f"  Epochs             : {total_epochs}")
print(f"  Timesteps / epoch  : {epoch_timesteps:,}")
print(f"  Cycle timesteps    : {cycle_timesteps:,}")
print(f"  Total timesteps    : {total_timesteps:,}")
print(f"  Model output dir   : {MODEL_DIR}")
print(f"  Log output dir     : {LOG_DIR}")
print("=" * 50)
print("Cell 6 done ✓")


## 7. Run Training

In [ ]:
import time

print(f"Starting SAC Foosball training")
print(f"{total_epochs} epochs × {epoch_timesteps:,} timesteps = {total_timesteps:,} total")
print("=" * 60)

start = time.time()

engine.train(
    total_epochs=total_epochs,
    epoch_timesteps=epoch_timesteps,
    cycle_timesteps=cycle_timesteps,
)

elapsed = time.time() - start
h, r = divmod(int(elapsed), 3600)
m, s = divmod(r, 60)
print(f"\nTraining complete ✓  ({h:02d}h {m:02d}m {s:02d}s)")


## 8. Save & Export Model
Zips the `models/` folder to `/kaggle/working/foosball_sac_models.zip` — downloadable from the Kaggle **Output** tab.

In [ ]:
import shutil, glob

shutil.make_archive(
    base_name="/kaggle/working/foosball_sac_models",
    format="zip",
    root_dir="/kaggle/working",
    base_dir="models",
)
zip_size = os.path.getsize("/kaggle/working/foosball_sac_models.zip") / 1e6
print(f"Archive: /kaggle/working/foosball_sac_models.zip  ({zip_size:.1f} MB)")

print("\nSaved checkpoints:")
for f in sorted(glob.glob(f"{MODEL_DIR}/**/*", recursive=True)):
    if os.path.isfile(f):
        print(f"  {f}")


## 9. Evaluate Trained Agent

In [ ]:
NUM_EVAL_EPISODES = 10

protagonist = agent_manager.get_frozen_best_models()[0]
eval_env = sac_foosball_env_factory()

episode_rewards = []
print(f"Evaluating for {NUM_EVAL_EPISODES} episodes ...\n")

for ep in range(NUM_EVAL_EPISODES):
    obs, _ = eval_env.reset()
    done = False
    total_reward = 0.0
    while not done:
        action = protagonist.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        total_reward += reward
        done = terminated or truncated
    episode_rewards.append(total_reward)
    print(f"  Episode {ep + 1:>2}/{NUM_EVAL_EPISODES}  reward: {total_reward:.2f}")

eval_env.close()

mean_r = sum(episode_rewards) / len(episode_rewards)
print(f"\nMean: {mean_r:.2f}  |  Min: {min(episode_rewards):.2f}  |  Max: {max(episode_rewards):.2f}")
